# Phase 1: Support-set study

## Section 1 - Purpose and overview

## Section 2 - Configuration

In [ ]:
# 2.1 Run once per environment, then leave it commented
# %pip install -q shapely pandas scikit-learn einops scipy opencv-python

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 2.2 Imports
import sys
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
import transformers

In [3]:
# 2.3 Paths and shared module
PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))   # makes src/perception importable from notebooks/

MODEL_DIR = PROJECT_ROOT / "models" / "c_radio_weights"
FRAMES_DIR = PROJECT_ROOT / "data" / "extracted_frames"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "phase_1_support_set_study"
BASELINE_DIR = PROJECT_ROOT / "outputs" / "baseline"
BASELINE_PROTOTYPES_PATH = BASELINE_DIR / "prototypes_v1-baseline.npz"             # saved by the v1-baseline pipeline
BASELINE_SIMILARITIES_PATH = BASELINE_DIR / "test_similarities_v1-baseline.npz"    # saved by the v1-baseline pipeline

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for name, path in {"MODEL_DIR": MODEL_DIR, "FRAMES_DIR": FRAMES_DIR, "SRC_DIR": SRC_DIR,
                   "BASELINE_PROTOTYPES_PATH": BASELINE_PROTOTYPES_PATH,
                   "BASELINE_SIMILARITIES_PATH": BASELINE_SIMILARITIES_PATH}.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"[{status}] {name}: {path}")

# Shared code (src/perception), the same used by the pipeline notebook
from perception.classes import CLASS_NAMES, HAND_LABEL, PIECE_LABELS, IGNORE_ID
from perception.features import FeatureExtractor
from perception.labels import sequence_of, check_annotation, rasterize_annotation, patch_label_grid
from perception.prototypes import build_prototypes, stack_prototypes, load_prototypes, assign_labels, classify_patches

[OK] MODEL_DIR: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\models\c_radio_weights
[OK] FRAMES_DIR: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\data\extracted_frames
[OK] SRC_DIR: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\src
[OK] BASELINE_PROTOTYPES_PATH: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\outputs\baseline\prototypes_v1-baseline.npz
[OK] BASELINE_SIMILARITIES_PATH: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\outputs\baseline\test_similarities_v1-baseline.npz


In [4]:
# 2.4 Hyperparameters
# Same values as the pipeline (phase_1_perception, Section 2.5): this study changes only the reference frames.

# --- Feature extraction ---
DEVICE = "cuda"
DTYPE = torch.bfloat16

# --- Patch labelling ---
MIN_LABEL_COVERAGE = 0.5

# --- Classification ---
CONFIDENCE_MARGIN = 0.03        # provisional, from the pipeline: not tuned in this study (margin study)

# --- Regression check (Section 3) ---
DETERMINISM_FRAME = "assembly_2_f0003"   # any validation frame
N_DETERMINISM_RUNS = 3                   # forward passes on the same frame

# --- Reproducibility ---
RANDOM_SEED = 0
np.random.seed(RANDOM_SEED)
_ = torch.manual_seed(RANDOM_SEED)

In [5]:
# 2.5 Frames and split
# Validation: errors are inspected and reference frames are chosen here.
# Test: held out until the end of all studies. This notebook never reads a test frame.
VALIDATION_SEQUENCES = {"assembly_2", "1_disassembled_controlled_caos"}
TEST_SEQUENCES = {"assembly_1"}
assert not VALIDATION_SEQUENCES & TEST_SEQUENCES


def frame_path(frame, suffix=".jpg"):
    """Path of a frame or of its annotation. Refuses test frames, so the test set cannot be read by mistake."""
    assert sequence_of(frame) not in TEST_SEQUENCES, f"{frame} belongs to the held-out test set"
    return FRAMES_DIR / f"{frame}{suffix}"


# Support set of the v1-baseline pipeline (phase_1_perception, Section 4.1): the starting point of the study
BASELINE_REFERENCE_FRAMES = [
    {"frame": "5_assembled_f0002",            "role": "assembled",                   "use_labels": PIECE_LABELS},
    {"frame": "caos_f0149",                   "role": "partially assembled",         "use_labels": PIECE_LABELS},
    {"frame": "3_disassembled_ordered_f0001", "role": "disassembled",                "use_labels": PIECE_LABELS - {"base"}},
    {"frame": "2_disassembled_caos_f0016",    "role": "disassembled, top-down view", "use_labels": PIECE_LABELS - {"base"}},
    {"frame": "Take_3_1_Base_f0099",          "role": "base only, upright",          "use_labels": {"base"}},
    {"frame": "Take_3_1_Base_f0131",          "role": "base only, lying",            "use_labels": {"base"}},
    {"frame": "caos_f0211",                   "role": "hands",                       "use_labels": {HAND_LABEL}},
]

VALIDATION_FRAMES = [
    {"frame": "assembly_2_f0003", "stage": "all pieces loose"},
    {"frame": "assembly_2_f0017", "stage": "body on base"},
    {"frame": "assembly_2_f0030", "stage": "hands: piece held in hand"},
    {"frame": "assembly_2_f0046", "stage": "face (eyes, nose, moustache, mouth)"},
    {"frame": "assembly_2_f0057", "stage": "ears mounted"},
    {"frame": "assembly_2_f0068", "stage": "arms mounted"},
    {"frame": "1_disassembled_controlled_caos_f0001", "stage": "disassembled, front view"},
    {"frame": "1_disassembled_controlled_caos_f0016", "stage": "disassembled, side view"},
]

# No leakage: reference frames never come from validation or test sequences
for ref in BASELINE_REFERENCE_FRAMES:
    assert sequence_of(ref["frame"]) not in VALIDATION_SEQUENCES | TEST_SEQUENCES, f"Leakage: {ref['frame']}"
for val in VALIDATION_FRAMES:
    assert sequence_of(val["frame"]) in VALIDATION_SEQUENCES, f"{val['frame']} is not a validation frame"
    assert frame_path(val["frame"]).exists(), f"Missing frame: {val['frame']}"

print(f"{len(BASELINE_REFERENCE_FRAMES)} baseline reference frames, {len(VALIDATION_FRAMES)} validation frames")

7 baseline reference frames, 8 validation frames


In [6]:
# 2.6 Environment
print(f"Python       : {platform.python_version()}")
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")

Python       : 3.10.0
PyTorch      : 2.5.1+cu121
Transformers : 5.17.0
CUDA         : True
GPU          : NVIDIA GeForce RTX 4060 Laptop GPU


## Section 3 - Regression check against v1-baseline

In [ ]:
# 3.1 Loading the model (shared FeatureExtractor, same settings as the pipeline)
extractor = FeatureExtractor(MODEL_DIR, device=DEVICE, dtype=DTYPE)
PATCH_SIZE = extractor.patch_size
print(f"Model loaded, patch size {PATCH_SIZE}x{PATCH_SIZE}")

Loading weights:   0%|          | 0/330 [00:00<?, ?it/s]

In [ ]:
# 3.2 Determinism: does the same frame give the same features twice?
# If yes, the module can be compared with v1-baseline for exact equality.
image = Image.open(frame_path(DETERMINISM_FRAME)).convert("RGB")
runs = [extractor.extract(image)[0] for _ in range(N_DETERMINISM_RUNS)]
run_diffs = [float(np.abs(run - runs[0]).max()) for run in runs[1:]]

print(f"Frame: {DETERMINISM_FRAME}, {N_DETERMINISM_RUNS} forward passes")
print(f"Max absolute feature difference vs run 1: {run_diffs}")
print(f"Typical feature magnitude (median |value|): {np.median(np.abs(runs[0])):.4f}")
print("Deterministic: bitwise identical runs" if max(run_diffs) == 0 else
      "NOT bitwise deterministic: compare with a tolerance above the run-to-run difference")

In [ ]:
# 3.3 Prototypes: module vs v1-baseline file
# Same 7 reference frames, rebuilt with src/perception. Patch counts do not involve the GPU,
# so they must match exactly; prototype vectors are compared numerically.
for ref in BASELINE_REFERENCE_FRAMES:
    ref["image_path"], ref["json_path"] = frame_path(ref["frame"]), frame_path(ref["frame"], ".json")
    check_annotation(ref["image_path"], ref["json_path"])
    image = Image.open(ref["image_path"]).convert("RGB")
    ref["features"], ref["resized_size"] = extractor.extract(image)
    pixel_map = rasterize_annotation(ref["json_path"], ref["use_labels"], ref["resized_size"])
    ref["patch_labels"], _ = patch_label_grid(pixel_map, PATCH_SIZE, MIN_LABEL_COVERAGE)

prototypes, patch_counts = build_prototypes(BASELINE_REFERENCE_FRAMES)
prototype_matrix = stack_prototypes(prototypes)

baseline = load_prototypes(BASELINE_PROTOTYPES_PATH)
assert baseline["class_names"] == CLASS_NAMES, "Class order differs from v1-baseline"

prototype_check = pd.DataFrame({
    "patches (v1)": baseline["patch_counts"],
    "patches (module)": [patch_counts[name] for name in CLASS_NAMES],
    "max |difference|": np.abs(prototype_matrix - baseline["prototypes"]).max(axis=1),
    "cosine(v1, module)": (prototype_matrix * baseline["prototypes"]).sum(axis=1),
}, index=CLASS_NAMES)
display(prototype_check)

print(f"Patch counts identical: {(prototype_check['patches (v1)'] == prototype_check['patches (module)']).all()}")
print(f"Prototypes bitwise identical: {np.array_equal(prototype_matrix, baseline['prototypes'])}")
print(f"Largest difference: {prototype_check['max |difference|'].max():.2e}, "
      f"lowest cosine: {prototype_check['cosine(v1, module)'].min():.8f}")

In [ ]:
# 3.4 Validation frames: similarities and labels, module vs v1-baseline
# The v1-baseline file stores raw similarities; labels are recomputed from them with the same margin.
baseline_similarities = np.load(BASELINE_SIMILARITIES_PATH)

rows = []
for val in VALIDATION_FRAMES:
    image = Image.open(frame_path(val["frame"])).convert("RGB")
    features, _ = extractor.extract(image)
    labels, similarities = classify_patches(features, prototype_matrix, CONFIDENCE_MARGIN)

    similarities_v1 = baseline_similarities[f"{val['frame']}__similarities"]
    labels_v1 = assign_labels(similarities_v1, CONFIDENCE_MARGIN)
    rows.append({
        "frame": val["frame"],
        "max |similarity difference|": float(np.abs(similarities - similarities_v1).max()),
        "labels identical (%)": 100 * float((labels == labels_v1).mean()),
        "patches with a different label": int((labels != labels_v1).sum()),
    })

regression_table = pd.DataFrame(rows).set_index("frame")
display(regression_table)
print(f"All labels identical on the {len(VALIDATION_FRAMES)} validation frames: "
      f"{(regression_table['patches with a different label'] == 0).all()}")